# ISMN stations → NISAR GCOV → **ONE MAAP DPS job** (all frames, 50 m sigma0)

This controller searches NISAR GCOV acquisitions from **June 1, 2026 through today**, over all station coordinates in `ismn_gt50_coords_points_june.csv`. It creates **one deduplicated scene manifest**, uploads that CSV to your private MAAP workspace S3 bucket, then makes exactly **one `maap.submitJob(...)` call** to the existing `nisar-gcov-ease2-v1` algorithm.

The single DPS worker sequentially downloads and processes each GCOV HDF5. It does **not** clip SAR to an AOI; it writes separately named full-frame GeoTIFFs in the same DPS `output/` directory. Each source granule is removed after processing to limit scratch usage.

> **Registration requirement:** GitHub `run_maap.sh` must have its new manifest-CSV dispatch enabled, and MAAP `nisar-gcov-ease2-v1:main` must be **re-registered/rebuilt** from that commit. Updating GitHub does not update an already-built DPS image. Re-registration replaces the existing registered build of that algorithm name, but remains backwards compatible with individual `.h5` inputs.
>
> **Operational limit:** MAAP DPS ordinarily has a **24-hour single-job runtime limit**. This notebook guarantees **one job submission**, not that an arbitrarily large frame collection can complete before the limit. Outputs are generally staged by DPS when the job finishes; an interrupted/failed job may have incomplete or triaged outputs.

**Collection selection:** `NISAR_L2_GCOV_PROVISIONAL_V1` and `NISAR_L2_GCOV_V1`, never BETA or urgent-response. Identical acquisitions prefer the validated release, then highest CRID/counter. A search bin is for finding candidates, **not clipping data**.

**Execution:** Run cells top to bottom with `SUBMIT_JOB=False` first. Inspect `granule_manifest.csv`. After re-registering MAAP and confirming input order, set `SUBMIT_JOB=True`, `CONFIRM_MAAP_BATCH_BUILD=True` and `CONFIRM_MAAP_INPUT_ORDER=True`. Only then does the last submission cell launch the single DPS job.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone, date, timedelta
from urllib.parse import urlparse
import hashlib, json, math, re, time, warnings

import pandas as pd
import earthaccess
from shapely.geometry import Point, Polygon, box
from shapely.ops import unary_union
from maap.maap import MAAP
from IPython.display import display

maap = MAAP()

# ---- INPUTS AND DATES ----
CSV_PATH = Path('ismn_gt50_coords_points_june.csv')
START_DATE = '2026-06-01'                    # ALL acquisitions on/after June 1, 2026
END_DATE = datetime.now(timezone.utc).date().isoformat()  # dynamically current UTC date
COLLECTIONS = {
    'provisional': 'NISAR_L2_GCOV_PROVISIONAL_V1',
    'validated': 'NISAR_L2_GCOV_V1',
}
PASSES = {'A', 'D'}
FREQUENCY = 'frequencyA'                    # original NISAR L-band frequency A
POLARIZATION_FILTER = None                  # set to a filename polarization code to restrict

# CMR spatial/time queries. BINS ARE FOR DISCOVERY ONLY, NEVER FOR CROPPING.
SEARCH_BIN_DEG = 5.0                       # bins with ≥1 station, not a continent-wide bbox
SEARCH_DAYS = 60                          # date slices to avoid huge CMR responses
USE_CMR_CACHE = True
REFRESH_CMR_CACHE = False

# ---- YOUR REGISTERED MAAP DPS ALGORITHM ----
ALGORITHM = 'nisar-gcov-ease2-v1'
VERSION = 'main'
WORKER_QUEUE = 'maap-dps-worker-32vcpu-64gb'  # choose an available MAAP queue
BATCH_IDENTIFIER = 'ISMN_NISAR_GCOV_JUNE2026_EASE2_50M_SIGMA0_SINGLEJOB'

EASE2_RES = '50'
BACKSCATTER = 'sigma0'
MODE = 'enhanced_lee'                       # retain 7x7 filter from supplied Indonesia notebook
FILTER_WINDOW = '7'
LIA = 'no'                                 # change to yes after DEM credentials test
CLIP_TO_AOI = 'no'                         # MUST remain 'no' to keep complete GCOV scenes

# ---- SAFE SUBMISSION SETTINGS ----
SUBMIT_JOB = False                         # DRY RUN by default; exactly ONE job if True
CONFIRM_MAAP_BATCH_BUILD = False           # must rebuild MAAP algorithm after GitHub updates
CONFIRM_MAAP_INPUT_ORDER = False           # verify registered input order before submitting

PRIVATE_CANDIDATES = [Path('/my-private-bucket'), Path.home()/'my-private-bucket',
                      Path('/projects/my-private-bucket')]
PRIVATE = next((p for p in PRIVATE_CANDIDATES if p.exists()), None)
if PRIVATE is None:
    raise RuntimeError('MAAP private bucket not mounted. Run in MAAP ADE, or modify PRIVATE_CANDIDATES.')
OUT = PRIVATE / 'ISMN_NISAR_GCOV_EASE2_50M_SIGMA0' / BATCH_IDENTIFIER
CACHE_DIR = OUT / 'cmr_cache'
OUT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)
MANIFEST_CSV = OUT / 'granule_manifest.csv'
MATCHES_CSV = OUT / 'station_granule_matches.csv'
ONE_JOB_RECORD = OUT / 'one_job_submission.json'

assert CSV_PATH.is_file(), f'Upload station CSV alongside the notebook: {CSV_PATH.resolve()}'
assert date.fromisoformat(START_DATE) <= date.fromisoformat(END_DATE)
assert CLIP_TO_AOI == 'no', 'This notebook is for FULL frames only'
assert EASE2_RES == '50' and BACKSCATTER == 'sigma0'
assert SEARCH_DAYS >= 1 and SEARCH_BIN_DEG > 0
print('Date window:', START_DATE, '→', END_DATE, '(UTC; both dates included)')
print('Collections:', list(COLLECTIONS.values()))
print('Algorithm:', ALGORITHM, ':', VERSION)
print('Processing:', EASE2_RES, 'm EASE2,', BACKSCATTER, MODE, FILTER_WINDOW+'×'+FILTER_WINDOW,
      'LIA='+LIA, 'clip='+CLIP_TO_AOI)
print('Saved manifests and ledger:', OUT)
print('SUBMIT_JOB:', SUBMIT_JOB, '(exactly one DPS job when enabled)')

In [ ]:
# Read and validate the EXACT coordinates from the provided CSV.
stations = pd.read_csv(CSV_PATH, dtype={'network': str, 'station': str})
required = ['network', 'station', 'lat', 'lon', 'n_points_june_onwards']
missing = sorted(set(required) - set(stations.columns))
if missing:
    raise ValueError(f'Missing station CSV columns: {missing}')
for key in ('lat', 'lon', 'n_points_june_onwards'):
    stations[key] = pd.to_numeric(stations[key], errors='raise')
if stations[required].isna().any().any():
    raise ValueError('Station CSV has missing required values')
if not stations.lat.between(-90, 90).all() or not stations.lon.between(-180, 180).all():
    raise ValueError('Invalid station lat/lon')
if stations.duplicated(['network', 'station']).any():
    raise ValueError('Duplicate network/station names; need unique station identifiers')

stations['station_id'] = stations.network.astype(str) + '::' + stations.station.astype(str)
source_sha256 = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()

# All geographic bins containing at least one station. Searching entire bins
# captures ANY granule containing a station, then a more precise point-in-footprint
# check removes irrelevant frames. This does not clip any SAR pixels.
def station_bin(lat, lon):
    return (int(math.floor(lon / SEARCH_BIN_DEG)), int(math.floor(lat / SEARCH_BIN_DEG)))

bin_to_sites = defaultdict(list)
site_by_id = {}
for rec in stations.to_dict('records'):
    station = {k: rec[k] for k in required}
    station['station_id'] = rec['station_id']
    bin_to_sites[station_bin(rec['lat'], rec['lon'])].append(station)
    site_by_id[rec['station_id']] = station

SEARCH_BINS = []
for (ix, iy), sites in sorted(bin_to_sites.items()):
    west, south = ix*SEARCH_BIN_DEG, iy*SEARCH_BIN_DEG
    bbox = (max(-180., west), max(-90., south), min(180., west+SEARCH_BIN_DEG),
            min(90., south+SEARCH_BIN_DEG))
    SEARCH_BINS.append((ix, iy, bbox, sites))

print(f'Stations: {len(stations)} | distinct search bins: {len(SEARCH_BINS)}')
print('Networks:', stations.network.value_counts().to_dict())
print('Station geographic range: lon', (stations.lon.min(), stations.lon.max()),
      'lat', (stations.lat.min(), stations.lat.max()))
display(stations.head(10))


In [ ]:
# CMR metadata helpers: filename parsing, physical geometry, HDF5 URL selection.
# Collection name (NOT filename prefix) determines provisional vs validated.
def granule_name(g):
    meta = g.get('meta', {})
    umm = g.get('umm', {})
    candidates = [meta.get('native-id'), meta.get('producer-granule-id'),
                  meta.get('title'), umm.get('GranuleUR')]
    for raw in candidates:
        if not raw:
            continue
        name = str(raw).split('?')[0].rstrip('/').split('/')[-1]
        name = re.sub(r'\.(h5|hdf5)$', '', name, flags=re.I)
        if name.startswith('NISAR_') and '_GCOV_' in name:
            return name
    return None


def parse_gcov_name(name):
    # NISAR_L2_PR_GCOV_cycle_track_pass_frame_mode_pol_source_start_end_CRID_*_*_*_counter
    parts = name.split('_')
    if len(parts) != 18 or parts[0] != 'NISAR' or parts[1] != 'L2' or parts[3] != 'GCOV':
        return None
    try:
        assert parts[6] in ('A', 'D')
        assert re.fullmatch(r'\d{8}T\d{6}', parts[11])
        assert re.fullmatch(r'\d{8}T\d{6}', parts[12])
        assert re.fullmatch(r'[A-Z]\d{5}', parts[13], re.I)
        return dict(cycle=int(parts[4]), track=int(parts[5]), pass_dir=parts[6],
                    frame=int(parts[7]), bw_mode=parts[8], polarization=parts[9],
                    acquisition_source=parts[10], date=parts[11][:8],
                    start_time=parts[11], end_time=parts[12], crid=parts[13],
                    crid_number=int(parts[13][1:]), coverage=parts[15],
                    counter=int(parts[17]))
    except (ValueError, AssertionError, IndexError):
        return None


def cmr_geometry(g):
    # Prefer actual granule polygons, then CMR bounding rectangles/boxes.
    umm = g.get('umm', {})
    geom = (umm.get('SpatialExtent', {}).get('HorizontalSpatialDomain', {})
            .get('Geometry', {}))
    actual = []
    for item in geom.get('GPolygons', []) or []:
        pts = item.get('Boundary', {}).get('Points', []) or []
        xy = [(float(p['Longitude']), float(p['Latitude'])) for p in pts
              if 'Longitude' in p and 'Latitude' in p]
        if len(xy) >= 3:
            polygon = Polygon(xy)
            if not polygon.is_valid:
                polygon = polygon.buffer(0)
            if not polygon.is_empty:
                actual.append(polygon)
    if actual:
        return unary_union(actual), 'polygon'
    rects = []
    for b in geom.get('BoundingRectangles', []) or []:
        try:
            rects.append(box(float(b['WestBoundingCoordinate']), float(b['SouthBoundingCoordinate']),
                             float(b['EastBoundingCoordinate']), float(b['NorthBoundingCoordinate'])))
        except (KeyError, TypeError, ValueError):
            continue
    if rects:
        return unary_union(rects), 'bounding_rectangle'
    for bounds in g.get('meta', {}).get('boxes', []) or []:
        try:
            s, w, n, e = map(float, bounds.split())
            return box(w, s, e, n), 'cmr_box_fallback'
        except (AttributeError, ValueError):
            continue
    return None, 'missing'


def gcov_hdf5_url(g, name):
    urls = []
    for access in ('direct', 'external'):
        try:
            urls.extend(g.data_links(access=access))
        except (TypeError, ValueError, AttributeError):
            pass
    for item in g.get('umm', {}).get('RelatedUrls', []) or []:
        if isinstance(item, dict) and item.get('URL'):
            urls.append(item['URL'])
    candidates = []
    for uri in dict.fromkeys(map(str, urls)):
        path = urlparse(uri).path
        filename = path.rstrip('/').split('/')[-1]
        if filename.lower() not in (name.lower()+'.h5', name.lower()+'.hdf5'):
            continue  # exclude *_QA_STATS.h5, quicklooks, ancillary files
        if uri.startswith('s3://'):
            priority = 0
        elif uri.startswith('https://'):
            priority = 1
        else:
            priority = 2
        candidates.append((priority, uri))
    candidates.sort()
    return candidates[0][1] if candidates else None


def dedup_identity(item):
    # Same physical GCOV acquisition across processor releases/CRIDs.
    return tuple(item[key] for key in ('cycle','track','pass_dir','frame',
                 'bw_mode','polarization','acquisition_source','start_time','end_time'))

print('CMR helpers loaded; full-frame processing: no bbox passed to MAAP.')


In [ ]:
# Discover provisional + validated GCOV over station-containing bins and 60-day
# time slices. All queries are cached (per date, collection, bin & CSV hash).
# Re-running this cell is safe; no MAAP jobs are submitted here.
start = date.fromisoformat(START_DATE)
end_exclusive = date.fromisoformat(END_DATE) + timedelta(days=1)
all_granules = {}               # exact filename -> full granule metadata
station_hits = defaultdict(dict)  # filename -> station_id -> match record
query_stats = Counter()

for maturity, collection in COLLECTIONS.items():
    t0 = start
    while t0 < end_exclusive:
        t1 = min(t0 + timedelta(days=SEARCH_DAYS), end_exclusive)
        temporal = (f'{t0.isoformat()}T00:00:00Z', f'{t1.isoformat()}T00:00:00Z')
        for ix, iy, search_bbox, sites in SEARCH_BINS:
            params = dict(collection=collection, temporal=list(temporal), bbox=list(search_bbox),
                          stations_sha256=source_sha256, bin_size=SEARCH_BIN_DEG,
                          matching_version=3, polarization_filter=POLARIZATION_FILTER)
            sig = hashlib.sha256(json.dumps(params, sort_keys=True).encode()).hexdigest()[:20]
            cache_path = CACHE_DIR / f'{maturity}_{t0:%Y%m%d}_bin{ix}_{iy}_{sig}.json'
            if USE_CMR_CACHE and not REFRESH_CMR_CACHE and cache_path.exists():
                cache = json.loads(cache_path.read_text())
                records = cache['records'] if cache.get('params') == params else None
            else:
                records = None
            if records is None:
                # Earthaccess/CMR searches the *intersecting full granules*;
                # site-in-footprint filtering happens below, not in processing.
                results = earthaccess.search_data(short_name=collection,
                    temporal=temporal, bounding_box=search_bbox, count=-1)
                query_stats['live_queries'] += 1
                records = []
                for g in results:
                    name = granule_name(g)
                    meta = parse_gcov_name(name) if name else None
                    if meta is None:
                        query_stats['unparsed_granules'] += 1
                        continue
                    if meta['pass_dir'] not in PASSES:
                        continue
                    if POLARIZATION_FILTER is not None and meta['polarization'] != POLARIZATION_FILTER:
                        continue
                    # Enforce acquisition date from filename in addition to CMR temporal search.
                    acq = date.fromisoformat(meta['date'][:4]+'-'+meta['date'][4:6]+'-'+meta['date'][6:8])
                    if not (start <= acq < end_exclusive):
                        continue
                    footprint, geometry_type = cmr_geometry(g)
                    if footprint is None:
                        query_stats['missing_cmr_footprints'] += 1
                        continue  # never guess that a broad search-bin hit covers a station
                    matches = [dict(station_id=site['station_id'],
                                    geometry_type=geometry_type,
                                    footprint_match=(geometry_type=='polygon'))
                               for site in sites
                               if footprint.covers(Point(site['lon'], site['lat']))]
                    if not matches:
                        continue
                    uri = gcov_hdf5_url(g, name)
                    if not uri:
                        query_stats['missing_main_gcov_hdf5_link'] += 1
                        continue
                    records.append(dict(granule=name, input_file=uri, collection=collection,
                                        maturity=maturity, matches=matches, **meta))
                if USE_CMR_CACHE:
                    # Save successful searches only; a failing CMR call is not cached.
                    cache_path.write_text(json.dumps({'params': params, 'records': records},
                                                      separators=(',', ':')))
            else:
                query_stats['cache_hits'] += 1
            for record in records:
                name = record['granule']
                candidate = {key: value for key, value in record.items() if key != 'matches'}
                if name not in all_granules:
                    all_granules[name] = candidate
                for site in record['matches']:
                    station_hits[name][site['station_id']] = site
                query_stats['station_frame_hits'] += len(record['matches'])
        print(f'{maturity:11s} {t0} — {t1-timedelta(days=1)} : '
              f'{len(all_granules):,} distinct granules | '
              f'{query_stats["live_queries"]} live queries, '
              f'{query_stats["cache_hits"]} cached')
        t0 = t1

print('CMR discovery finished:', dict(query_stats))
print('Unique GCOV filenames matching ≥1 station:', len(all_granules))
if not all_granules:
    warnings.warn('No matching granules. Inspect CMR access, date interval, collection names and station locations.')
if query_stats['missing_cmr_footprints']:
    warnings.warn(f'{query_stats["missing_cmr_footprints"]} hits had missing CMR geometry and were excluded '
                  'rather than mistakenly assigning a frame to an unrelated station.')


In [ ]:
# Build ONE row per distinct GCOV HDF5 scene and ONE row per station-scene pair.
# For the same physical acquisition: validated > provisional, then highest CRID/counter.
raw = list(all_granules.values())
variants_by_identity = defaultdict(list)
for variant in raw:
    variants_by_identity[dedup_identity(variant)].append(variant)
selected_by_acq = {}
for item in raw:
    key = dedup_identity(item)
    score = (1 if item['maturity'] == 'validated' else 0,
             int(item['crid_number']), int(item['counter']))
    old = selected_by_acq.get(key)
    if old is None or score > old[0]:
        selected_by_acq[key] = (score, item)

selected = [pair[1] for pair in selected_by_acq.values()]
selected.sort(key=lambda r: (r['date'], r['start_time'], r['track'], r['frame']))
scene_records = []
match_records = []
for item in selected:
    # Union station assignments from superseded versions of this SAME acquisition.
    identity = dedup_identity(item)
    all_sites_for_scene = {}
    for variant in variants_by_identity[identity]:
        all_sites_for_scene.update(station_hits[variant['granule']])
    for sid, match in sorted(all_sites_for_scene.items()):
        st = site_by_id[sid]
        match_records.append(dict(granule=item['granule'], maturity=item['maturity'],
            collection=item['collection'], date=item['date'], pass_dir=item['pass_dir'],
            track=item['track'], frame=item['frame'], station_id=sid,
            network=st['network'], station=st['station'], lat=st['lat'], lon=st['lon'],
            n_points_june_onwards=st['n_points_june_onwards'],
            geometry_type=match['geometry_type']))
    scene_records.append({**item, 'matched_stations':len(all_sites_for_scene),
                          'station_ids':' | '.join(sorted(all_sites_for_scene))})

manifest = pd.DataFrame(scene_records)
matches = pd.DataFrame(match_records)
manifest_columns = ['granule','maturity','collection','date','pass_dir','track','frame',
    'cycle','bw_mode','polarization','crid','input_file','matched_stations','station_ids']
match_columns = ['granule','maturity','date','pass_dir','track','frame',
    'station_id','network','station','lat','lon','n_points_june_onwards','geometry_type']
manifest = manifest.reindex(columns=manifest_columns)
matches = matches.reindex(columns=match_columns)
manifest.to_csv(MANIFEST_CSV, index=False)
matches.to_csv(MATCHES_CSV, index=False)
print('Station count:', len(stations), '| selected FULL GCOV frames:', len(manifest),
      '| station–frame matches:', len(matches),
      '| unique stations covered:', matches.station_id.nunique())
if len(manifest):
    print('Frames by maturity:', manifest.maturity.value_counts().to_dict())
    print('Frames by pass:', manifest.pass_dir.value_counts().to_dict())
    print('Frames by acquisition month:',manifest.date.str.slice(0,6).value_counts().sort_index().to_dict())
    display(manifest[['date','maturity','pass_dir','track','frame','matched_stations','granule']].head(25))
if len(matches):
    covered = set(matches.station_id)
    not_covered = stations.loc[~stations.station_id.isin(covered),['network','station','lat','lon']]
    print('Stations with no CMR match:',len(not_covered))
    if len(not_covered):
        display(not_covered.head(20))
print('Granule manifest:', MANIFEST_CSV)
print('Station coverage table:', MATCHES_CSV)


In [ ]:
# Preflight the registered MAAP algorithm and its positional schema.
# IMPORTANT: MAAP build images are snapshots, NOT automatically refreshed by a GitHub push.
response = maap.listAlgorithms()
response.raise_for_status() if hasattr(response, 'raise_for_status') else None
catalog = response.json().get('algorithms', [])
registered = [x for x in catalog if x.get('type') == ALGORITHM and x.get('version') == VERSION]
print('Algorithm registered:', bool(registered), f'{ALGORITHM}:{VERSION}')
if not registered:
    raise RuntimeError(f'{ALGORITHM}:{VERSION} not registered in MAAP')
print('Registration metadata (inspect before submitting):')
print(json.dumps(registered[0], indent=2, default=str)[:7000])
required_order = ['input_file', 'ease2_res', 'mode', 'filter_window', 'lia',
                  'frequency', 'backscatter', 'clip_to_aoi']
print('Required registered input order:', ' → '.join(required_order))
print('Manifest entries:', len(manifest), '| original stations:',len(stations))
if SUBMIT_JOB and not (CONFIRM_MAAP_INPUT_ORDER and CONFIRM_MAAP_BATCH_BUILD):
    raise RuntimeError('Re-register/rebuild your MAAP algorithm using the updated GitHub '
         'run_maap.sh, check input order, then set BOTH CONFIRM_... flags True.')

In [ ]:
# Upload the SINGLE manifest into your authorized private workspace S3 folder
# and submit EXACTLY ONE MAAP job (not a loop). No file has to be public.
# This needs a MAAP ADE environment with maap-py, boto3 and writable workspace bucket.
from urllib.parse import urlparse

bad = manifest.loc[~manifest.input_file.astype(str).str.startswith('s3://')]
if len(bad):
    display(bad[['granule','input_file']].head(15))
    raise RuntimeError(f'{len(bad)} GCOV scenes lack S3 data URLs. The one-job downloader '
                       'currently requires authenticated NISAR S3 URLs for each scene.')
if manifest.empty:
    raise RuntimeError('No matching GCOV scenes; nothing to submit.')
assert manifest.granule.is_unique
assert set(manifest.maturity) <= {'provisional','validated'}
print(f'One-job manifest: {len(manifest)} scenes from {len(stations)} stations')
print('Existing manifest:', MANIFEST_CSV)
print('DO NOT use the earlier per-granule submission notebook for this run.')

if not SUBMIT_JOB:
    print('DRY RUN: no S3 manifest uploaded, no MAAP job submitted.')
    print('After registering new GitHub batch support, set SUBMIT_JOB=True, '
          'CONFIRM_MAAP_BATCH_BUILD=True and CONFIRM_MAAP_INPUT_ORDER=True.')
else:
    if not (CONFIRM_MAAP_INPUT_ORDER and CONFIRM_MAAP_BATCH_BUILD):
        raise RuntimeError('Confirm rebuilt batch-capable MAAP image and input order first.')
    if ONE_JOB_RECORD.is_file():
        print('Existing single-job submission recorded:', ONE_JOB_RECORD)
        raise RuntimeError('Existing job record found. Inspect it before submitting '
                           'another MAAP job; remove/rename record ONLY if intentionally rerunning.')
    import boto3
    reply = maap.aws.workspace_bucket_credentials()
    if hasattr(reply, 'json'):
        reply = reply.json()
    cred = reply['credentials']
    available = reply['authorized_s3_paths']
    workspace = next((p for p in available if p.get('type') == 'workspace'
                      and p.get('access') == 'read_write'), None)
    if workspace is None:
        raise RuntimeError('MAAP returned no writable workspace S3 path')
    bucket = workspace['bucket']
    prefix = workspace.get('prefix','').strip('/')
    object_name = f'ISMN_NISAR_GCOV_EASE2_50M/{BATCH_IDENTIFIER}/granule_manifest.csv'
    key = f'{prefix}/{object_name}' if prefix else object_name
    # Credentials are short lived and intentionally NEVER printed or saved.
    session = boto3.Session(aws_access_key_id=cred['aws_access_key_id'],
                            aws_secret_access_key=cred['aws_secret_access_key'],
                            aws_session_token=cred['aws_session_token'],
                            region_name='us-west-2')
    s3 = session.client('s3')
    s3.upload_file(str(MANIFEST_CSV), bucket, key)
    info = s3.head_object(Bucket=bucket, Key=key)
    if info['ContentLength'] <= 0:
        raise RuntimeError('Uploaded manifest appears empty')
    manifest_uri = f's3://{bucket}/{key}'
    print('Staged MANIFEST (one File Input):', manifest_uri)
    print('Submitting ONE MAAP DPS job processing',len(manifest),'scenes...')
    kwargs = dict(identifier=BATCH_IDENTIFIER, algo_id=ALGORITHM, version=VERSION,
        queue=WORKER_QUEUE, input_file=manifest_uri,
        ease2_res='50', mode=MODE, filter_window=FILTER_WINDOW,
        lia=LIA, frequency=FREQUENCY, backscatter='sigma0', clip_to_aoi='no')
    # Exactly ONE submission. The job downloads its own GCOV scenes sequentially.
    result = maap.submitJob(**kwargs)
    job_id = getattr(result,'id',None)
    if not job_id:
        raise RuntimeError(f'MAAP did not return a job ID; inspect the jobs UI before retry: {result}')
    record = dict(job_id=str(job_id), input_file=manifest_uri, manifest_count=len(manifest),
                  station_count=len(stations), created_utc=datetime.now(timezone.utc).isoformat(),
                  settings={k: v for k,v in kwargs.items() if k != 'input_file'})
    ONE_JOB_RECORD.write_text(json.dumps(record,indent=2)+'\n')
    print('ONE MAAP JOB accepted:',job_id)
    print('Submission record saved:',ONE_JOB_RECORD)
    print('All 50 m sigma0 outputs will be under THIS job output/ if it finishes successfully.')

In [ ]:
# Check status of the ONE submitted MAAP job (run any time).
if ONE_JOB_RECORD.is_file():
    record=json.loads(ONE_JOB_RECORD.read_text())
    print('Job ID:', record['job_id'], '| planned granules:',record['manifest_count'])
    print('Manifest:',record['input_file'])
    response=maap.getJobStatus(record['job_id'])
    print('Latest MAAP status:',getattr(response,'text',str(response))[:1500])
else:
    print('No job submitted yet; still in DRY RUN.')
print('Scene manifest:', MANIFEST_CSV)
print('Station-to-frame matching:', MATCHES_CSV)

## What this code guarantees / does not guarantee

- **One `maap.submitJob` call** and **one job ID**, with all selected granules processed sequentially inside that single DPS worker. It does not launch subjobs.
- **50 m EASE2 sigma0 in linear power**; `enhanced_lee` 7 × 7; frequencyA; optional `lia=no`; no AOI clipping.
- **Provisional / validated** selected from CMR collections; validated replaces provisional for identical acquisition identities. Actual CRID-aware masks are evaluated by the unchanged EASE2 processor.
- **Separate full-frame outputs** for each acquisition. Because files are processed one at a time, across-frame mosaicking is **not** performed inside this one-job version. This avoids holding dozens of HDF5 frames simultaneously.
- **Per-scene progress:** the worker saves `output/single_dps_job_scene_ledger.csv`, `output/single_dps_job_summary.json`, and a copy of the manifest, together with the individual GeoTIFF/PNG/MAT files.
- **MAAP runtime/storage:** A 24-hour default job limit may prevent all scenes from finishing. One DPS job is not a scalable substitute for many per-granule jobs. Scratch disk must hold the current full HDF5, native intermediate rasters, and *all outputs* until DPS collects them. If job fails or times out, output publishing may be incomplete.
- **No MAAP job submitted by generating this notebook.** You must set explicit execution flags and run inside MAAP after refreshing the algorithm registration.